# Multi-Scale Multi-Modal Data: Fine-Grained Correspondences

Two modalities with **complementary resolutions** (e.g. EEG-like vs fMRI-like):

- **Modality 1 (S1)**: **Good temporal resolution**, **poor spatial resolution** — e.g. (t_fine, x_coarse). Like EEG: many time points, fewer channels.
- **Modality 2 (S2)**: **Poor temporal resolution**, **good spatial resolution** — e.g. (t_coarse, x_fine). Like fMRI: few time points, many voxels.

**Critical sampling (Nyquist)**: Each modality is sampled at or up to the Nyquist criterion in its "good" dimension so that simple resampling does not recover the other modality's information:
- S1 carries high temporal frequencies (needs t_fine; cannot be inferred from S2's coarse time).
- S2 carries high spatial frequencies (needs x_fine; cannot be inferred from S1's coarse space).

We generate a shared driver and couple S2 to it; S1 is observed on (t_fine, x_coarse), S2 on (t_coarse, x_fine), with **explicit correspondence** maps between the two grids.

## 1. Imports and Nyquist / grid setup

Critical sampling: for grid spacing Δ (in time or space), Nyquist limit is f_max = 1/(2Δ). We set signal bandwidth so each modality is at Nyquist in its "good" dimension and bandlimited in its "bad" dimension.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from tqdm import tqdm

# --- Grid and Nyquist setup ---
# Modality 1 (S1): good time, poor space -> (n_t_fine, n_x_coarse)
# Modality 2 (S2): poor time, good space -> (n_t_coarse, n_x_fine)
# Driver U lives on full fine grid (n_t_fine, n_x_fine) for coupling; we observe S1 on coarse x, S2 on coarse t.

def make_grids_and_nyquist(t_extent, x_extent, n_t_fine, n_x_fine, n_t_coarse, n_x_coarse):
    """
    Build time and space grids. Return grids + Nyquist limits (for critical sampling).
    Nyquist: f_max = 1/(2*dx) or 1/(2*dt); in angular form omega_max = pi/dt, k_max = pi/dx.
    """
    t_fine = np.linspace(0, t_extent, n_t_fine)
    t_coarse = np.linspace(0, t_extent, n_t_coarse)
    x_fine = np.linspace(0, x_extent, n_x_fine)
    x_coarse = np.linspace(0, x_extent, n_x_coarse)
    dt_fine = t_extent / (n_t_fine - 1) if n_t_fine > 1 else t_extent
    dt_coarse = t_extent / (n_t_coarse - 1) if n_t_coarse > 1 else t_extent
    dx_fine = x_extent / (n_x_fine - 1) if n_x_fine > 1 else x_extent
    dx_coarse = x_extent / (n_x_coarse - 1) if n_x_coarse > 1 else x_extent
    # Angular Nyquist: omega_t <= pi/dt, k_space <= pi/dx
    omega_nyq_fine = np.pi / dt_fine
    omega_nyq_coarse = np.pi / dt_coarse
    k_nyq_fine = np.pi / dx_fine
    k_nyq_coarse = np.pi / dx_coarse
    return {
        't_fine': t_fine, 't_coarse': t_coarse, 'x_fine': x_fine, 'x_coarse': x_coarse,
        'dt_fine': dt_fine, 'dt_coarse': dt_coarse, 'dx_fine': dx_fine, 'dx_coarse': dx_coarse,
        'omega_nyq_t_fine': omega_nyq_fine, 'omega_nyq_t_coarse': omega_nyq_coarse,
        'k_nyq_x_fine': k_nyq_fine, 'k_nyq_x_coarse': k_nyq_coarse,
    }


def generate_base_signal(x, t, params):
    """Generates signal on grid (t, x) with phase components. Returns (T, X) array and phases."""
    X, T = np.meshgrid(x, t)
    out = np.zeros_like(X, dtype=float)
    phases = []
    for p in params:
        component_phase = p['k'] * X - p['omega'] * T + p['phi']
        out += p['A'] * np.sin(component_phase)
        phases.append(component_phase)
    return out, np.array(phases)


def kuramoto_ode(t_point, theta, t_grid, base_phases_interp, K, omega_prime):
    """Kuramoto-like ODE: dθ/dt = ω' + K * mean_field(sin(ψ - θ))."""
    psi = np.array([interp_func(t_point) for interp_func in base_phases_interp])
    mean_field = np.mean(np.sin(psi - theta[:, np.newaxis]), axis=1)
    return omega_prime + K * mean_field


## 2. Cross-modal correspondence (good-time vs good-space)

S1 is on (t_fine, x_coarse); S2 is on (t_coarse, x_fine). Correspondence:
- **Time**: each S2 time index k_c corresponds to a block of S1 (driver) time indices.
- **Space**: each S1 space index i_c corresponds to a block of S2 (driver) space indices.
We store coarse_to_fine slices so we know which fine (t,x) block each coarse (t,x) point represents.

In [ ]:
def build_cross_modal_correspondence(n_t_fine, n_x_fine, n_t_coarse, n_x_coarse):
    """
    S1 grid: (n_t_fine, n_x_coarse). S2 grid: (n_t_coarse, n_x_fine).
    Assume n_t_fine = n_t_coarse * r_t and n_x_fine = n_x_coarse * r_x (integer ratios).
    Returns:
      c2f_t: for each S2 time index k_c, slice of driver/S1 time indices
      c2f_x: for each S1 space index i_c, slice of driver/S2 space indices
    """
    r_t = n_t_fine // n_t_coarse
    r_x = n_x_fine // n_x_coarse
    c2f_t = [slice(k_c * r_t, min((k_c + 1) * r_t, n_t_fine)) for k_c in range(n_t_coarse)]
    c2f_x = [slice(i_c * r_x, min((i_c + 1) * r_x, n_x_fine)) for i_c in range(n_x_coarse)]
    return c2f_t, c2f_x


## 3. Generate S1 (good time / poor space) and S2 (poor time / good space)

- **Driver U** on full fine grid (t_fine, x_fine) with content up to Nyquist in both dimensions.
- **S1** (modality 1): U downsampled in **space** only — average U over each x_coarse block → shape (n_t_fine, n_x_coarse). Critically sampled in **time** (high ω); bandlimited in space (low k in effective content after spatial averaging).
- **S2** (modality 2): Kuramoto-coupled to U at each x_fine; ODE evaluated at **t_coarse** only → shape (n_t_coarse, n_x_fine). Critically sampled in **space** (high k); bandlimited in time (ω chosen so t_coarse is at Nyquist).

In [ ]:
def generate_cross_modal_dataset(grid_info, driver_params, s2_params, K, nyquist_scale=0.9):
    """
    Generate S1 (good time, poor space) and S2 (poor time, good space) from a shared driver U.
    driver_params: list of {A, k, omega, phi} for U on (t_fine, x_fine). Use omega <= nyquist_scale*omega_nyq_t_fine, k <= nyquist_scale*k_nyq_x_fine.
    s2_params: for coupled S2; use omega <= nyquist_scale*omega_nyq_t_coarse so S2 is critically sampled in time.
    """
    t_fine = grid_info['t_fine']
    t_coarse = grid_info['t_coarse']
    x_fine = grid_info['x_fine']
    x_coarse = grid_info['x_coarse']
    n_t_fine, n_x_fine = len(t_fine), len(x_fine)
    n_t_coarse, n_x_coarse = len(t_coarse), len(x_coarse)
    c2f_t, c2f_x = build_cross_modal_correspondence(n_t_fine, n_x_fine, n_t_coarse, n_x_coarse)

    # Driver U on full fine grid (critically sampled up to Nyquist)
    U, base_phases = generate_base_signal(x_fine, t_fine, driver_params)
    # U, base_phases: (n_t_fine, n_x_fine) and (n_components, n_t_fine, n_x_fine)

    # S1: good time, poor space — average U over each x_coarse block
    S1 = np.zeros((n_t_fine, n_x_coarse))
    for i_c in range(n_x_coarse):
        sl = c2f_x[i_c]
        S1[:, i_c] = np.mean(U[:, sl], axis=1)

    # S2: poor time, good space — at each x_fine solve ODE driven by U(., x_fine), eval at t_coarse
    omega_prime = np.array([p['omega'] for p in s2_params])
    initial_phases_s2 = np.array([p['phi'] for p in s2_params])
    amplitudes_s2 = np.array([p['A'] for p in s2_params])
    S2 = np.zeros((n_t_coarse, n_x_fine))
    for i_x in range(n_x_fine):
        base_phases_at_x = base_phases[:, :, i_x]
        base_phases_interp = [
            lambda t_eval, phase_series=series: np.interp(t_eval, t_fine, phase_series)
            for series in base_phases_at_x
        ]
        sol = solve_ivp(
            kuramoto_ode, [t_fine[0], t_fine[-1]], initial_phases_s2, t_eval=t_coarse,
            args=(t_fine, base_phases_interp, K, omega_prime), method='RK45',
        )
        S2[:, i_x] = np.sum(amplitudes_s2[:, np.newaxis] * np.sin(sol.y), axis=0)

    correspondence = {'coarse_to_fine_t': c2f_t, 'coarse_to_fine_x': c2f_x}
    grids = {'S1': (t_fine, x_coarse), 'S2': (t_coarse, x_fine), 'driver': (t_fine, x_fine)}
    return S1, S2, U, grids, correspondence


## 4. Run with Nyquist-respecting parameters

In [ ]:
# Grid sizes: S1 = (good time, poor space), S2 = (poor time, good space)
t_extent, x_extent = 20.0, 10.0
n_t_fine, n_x_fine = 128, 32
n_t_coarse, n_x_coarse = 8, 8
# So S1: 128 x 8 (good time, poor space), S2: 8 x 32 (poor time, good space)

grid_info = make_grids_and_nyquist(t_extent, x_extent, n_t_fine, n_x_fine, n_t_coarse, n_x_coarse)
# Choose driver params up to Nyquist on fine grid (so resampling loses info)
omega_nyq_t = grid_info['omega_nyq_t_fine']
k_nyq_x = grid_info['k_nyq_x_fine']
scale = 0.85  # stay below Nyquist
driver_params = [
    {'A': 1.0, 'k': 0.3 * k_nyq_x, 'omega': 0.4 * omega_nyq_t, 'phi': 0},
    {'A': 0.5, 'k': 0.6 * k_nyq_x, 'omega': 0.8 * omega_nyq_t, 'phi': np.pi / 2},
    {'A': 0.2, 'k': scale * k_nyq_x, 'omega': scale * omega_nyq_t, 'phi': np.pi},
]
# S2 coupled dynamics: temporal content limited so t_coarse is sufficient (critical sampling in time)
omega_nyq_t_coarse = grid_info['omega_nyq_t_coarse']
s2_params = [
    {'A': 1.0, 'omega': 0.5 * omega_nyq_t_coarse, 'phi': 0.1},
    {'A': 0.8, 'omega': 0.85 * omega_nyq_t_coarse, 'phi': 0.2},
]
K = 2.5

S1, S2, U, grids, correspondence = generate_cross_modal_dataset(
    grid_info, driver_params, s2_params, K
)

print("S1 (good time, poor space):", S1.shape, "=", grids['S1'][0].shape[0], "t x", grids['S1'][1].shape[0], "x")
print("S2 (poor time, good space):", S2.shape, "=", grids['S2'][0].shape[0], "t x", grids['S2'][1].shape[0], "x")
print("Driver U (full fine):", U.shape)
print("Correspondence: S2 time block 0 -> driver t indices", correspondence['coarse_to_fine_t'][0])
print("Correspondence: S1 space block 0 -> driver x indices", correspondence['coarse_to_fine_x'][0])


## 5. Save dataset with correspondence (for learning)

We save S1, S2, grids, and the cross-modal correspondence (coarse_to_fine_t, coarse_to_fine_x) so that a model can align the two modalities.

In [ ]:
def save_cross_modal_dataset(filepath, S1, S2, grids, correspondence, K):
    """Save cross-modal dataset. S1=(t_fine,x_coarse), S2=(t_coarse,x_fine)."""
    c2f_t = np.array([[s.start, s.stop] for s in correspondence['coarse_to_fine_t']])
    c2f_x = np.array([[s.start, s.stop] for s in correspondence['coarse_to_fine_x']])
    np.savez_compressed(
        filepath,
        S1=S1, S2=S2,
        t_fine=grids['S1'][0], x_coarse=grids['S1'][1],
        t_coarse=grids['S2'][0], x_fine=grids['S2'][1],
        coarse_to_fine_t=c2f_t, coarse_to_fine_x=c2f_x,
        K=np.array(K),
    )
    print(f"Saved to {filepath}")


save_cross_modal_dataset("multiscale_multimodal_sample.npz", S1, S2, grids, correspondence, K)


## 6. Quick visualization

Plot S1 (good time, poor space) and S2 (poor time, good space) to show complementary resolutions.

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 1, figsize=(10, 6))
t_s1, x_s1 = grids['S1'][0], grids['S1'][1]
axs[0].imshow(S1, aspect='auto', origin='lower', extent=[x_s1[0], x_s1[-1], t_s1[0], t_s1[-1]], cmap='viridis')
axs[0].set_title('S1: good time resolution, poor spatial (%d t × %d x)' % (S1.shape[0], S1.shape[1]))
axs[0].set_xlabel('x')
axs[0].set_ylabel('t')

t_s2, x_s2 = grids['S2'][0], grids['S2'][1]
axs[1].imshow(S2, aspect='auto', origin='lower', extent=[x_s2[0], x_s2[-1], t_s2[0], t_s2[-1]], cmap='plasma')
axs[1].set_title('S2: poor time resolution, good spatial (%d t × %d x)' % (S2.shape[0], S2.shape[1]))
axs[1].set_xlabel('x')
axs[1].set_ylabel('t')
plt.tight_layout()
plt.show()
print("Cross-modal data: each modality critically sampled in its strong dimension; correspondence stored for alignment.")
